# 36. The bucket no order ever lands in

**Track:** debug-the-ai | **Difficulty:** easy | **Tags:** when otherwise, case when, bucketing, conditional logic

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("036-when-chain-order")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "orders": """
UEFSMRUEFaAaFa4NTBWkAxUAEgAAkA0IYeoABQEAYg0IAGMNCABkDQgAZQ0IAGYNCABnDQgAaA0IAGkNCABqDQgAaw0IAGwNCABt
DQgAbg0IAG8NCABwDQgAcQ0IAHINCABzDQgAdA0IAHUNCAB2DQgAdw0IAHgNCAB5DQgAeg0IAHsNCAB8DQgAfQ0IAH4NCAB/DQgA
gA0IAIENCACCDQgAgw0IAIQNCACFDQgAhg0IAIcNCACIDQgAiQ0IAIoNCACLDQgAjA0IAI0NCACODQgAjw0IAJANCACRDQgAkg0I
AJMNCACUDQgAlQ0IAJYNCACXDQgAmA0IAJkNCACaDQgAmw0IAJwNCACdDQgAng0IAJ8NCACgDQgAoQ0IAKINCACjDQgApA0IAKUN
CACmDQgApw0IAKgNCACpDQgAqg0IAKsNCACsDQgArQ0IAK4NCACvDQgAsA0IALENCACyDQgAsw0IALQNCAC1DQgAtg0IALcNCAC4
DQgAuQ0IALoNCAC7DQgAvA0IAL0NCAC+DQgAvw0IAMANCADBDQgAwg0IAMMNCADEDQgAxQ0IAMYNCADHDQgAyA0IAMkNCADKDQgA
yw0IAMwNCADNDQgAzg0IAM8NCADQDQgA0Q0IANINCADTDQgA1A0IANUNCADWDQgA1w0IANgNCADZDQgA2g0IANsNCADcDQgA3Q0I
AN4NCADfDQgA4A0IAOENCADiDQgA4w0IAOQNCADlDQgA5g0IAOcNCADoDQgA6Q0IAOoNCADrDQgA7A0IAO0NCADuDQgA7w0IAPAN
CADxDQgA8g0IAPMNCAD0DQgA9Q0IAPYNCAD3DQgA+A0IAPkNCAD6DQgA+w0IAPwNCAD9DQgA/g0IAP8NCAgA6wAFAQABDQgAAg0I
AAMNCAAEDQgABQ0IAAYNCAAHDQgACA0IAAkNCAAKDQgACw0IAAwNCAANDQgADg0IAA8NCAAQDQgAEQ0IABINCAATDQgAFA0IABUN
CAAWDQgAFw0IABgNCAAZDQgAGg0IABsNCAAcDQgAHQ0IAB4NCAAfDQgAIA0IACENCAAiDQgAIw0IACQNCAAlDQgAJg0IACcNCAAo
DQgAKQ0IACoNCAArDQgALA0IAC0NCAAuDQgALw0IADANCDwx6wAAAAAAADLrAAAAAAAAFQAVwgMVygMsFaQDFRAVBhUGHBgIMusA
AAAAAAAYCGHqAAAAAAAAFgAoCDLrAAAAAAAAGAhh6gAAAAAAABERAAAA4QHw4AMAAACkAwEINwABAgMEBQYHCAkKCwwNDg8QERIT
FBUWFxgZGhscHR4fICEiIyQlJicoKSorLC0uLzAxMjM0NTY3ODk6Ozw9Pj9AQUJDREVGR0hJSktMTU5PUFFSU1RVVldYWVpbXF1e
X2BhYmNkZWZnaGlqa2xtbm9wcXJzdHV2d3h5ent8fX5/gIGCg4SFhoeIiYqLjI2Oj5CRkpOUlZaXmJmam5ydnp+goaKjpKWmp6ip
qqusra6vsLGys7S1tre4ubq7vL2+v8DBwsPExcbHyMnKy8zNzs/Q0QAAAAAAABUEFbQQFboQTBWkAxUAEgAAmgiwAAAPQkEAAAwD
TQAAAKBYAAAAnDkAAAB5VgAAAIRlAAAAR9wAAARiYgAAAIg3AR7wlZ0AADjAMgAAANVXAAABMLAAAABFDAAAAVijAAABSKMAAAPC
jQAAhNNxAAABhqAAAAAwKwAAC8+/AAANnk8AAAD7rQAACGF9AABTJOEAAAF4+gAAAS4tAAAAZs8AAAPP0wAACoH9AAAAaycAAF9M
FgAAAJ2+AAAA1c8AAACQ6gAAANYNAAAA+YYAAAfqkQAAAKHaAAAAOAEo8GXLVAAACQPBAAAAw6IAAAD+wQAACa2lAAAIyRMAAAje
FQAAAGcxAAAHobwAACLSfAAAcNw0AAABKooAAIlOhwAAAIHFAAAAIFYAAABjIwAAAXrcAAAJiQYAAAElAgAABb13AAAFjGohMfCG
PwAAA707AAAJg3kAAAW60AAADQ6zAAAMaeAAAABACwAAAWYZAAAAzlMAABHAdgAAPx6oAAAAKv4AAAFyDQAAev4SAAAA0RAAAAC/
3gAAAWqpAAAANWsAAAFZAAAADjL7AAAAg/kAAAChzQAADXpkAAABR8MAAAC5kAAADGkQAAAJ/eAAAA1yAWT0XgHBvQAADG7pAAAH
rqEAAArhFgAAAVrqAAAEFekAAH6vZwAAAEgCAAA1dbAAAArcxgAAAYafAAAdk+4AAABTcQAATczhAAANCm8AAAFz3gAAAUHdAAAF
y4oAAACKuAAAAXGBAAAAWvkAAADrjgAAAEc+AAABNe0AAANHvwAACmCjAAAN0s4AAAEqwgAABo+LAAAAofcAAADMsAAAAOvoAAAB
giIAAAvIAQAAA1vUAAADOnsAAAA8YgAAI+/jAAAAKccAAA0UUAAAAOpTAAAEDnAAAAAj9wAACh4OAAABepwAAAAjzQAABcOSAAAA
4O0AAF00/AAAAJDfAAANAzMAAAmmFgAAAWBFAAAAmvcAAAC1CwAAAorVAAABfEUAAFRMnwAACxGEAAAAZUYAAGELWAAAUD1YAABb
7HcAAADWJwAAATsiAAAHUVwAAA2KOAAAAV9TAAABbgAAAA9U/QAABvchi/B9Q4AAAAEWRQAAAEyIAAAAqaMAAAY4zQAAAVA2AAAL
kKQAAAsPYgAAAIXQAAABS2gAAACDeQAAAVVlAAAANyQAAAFBPAAAB9siAAAFCPYAAATtiwAAAVs2AAADT4oAAAGT6gAAAIjNAAAA
fswAAAEmFAAADbefAAACR50AAAA7QVOgf40AAAG1pgAACWwQAAAAp7IAAABDBAAAA/ipAAAFeM8AAAh7bAAAhVRhEfBHFtsAAAL6
/QAAcuxIAAAAiqcAAAomRgAAKLgrAAABMuMAAAAhHgAACEfKAAAPQkAAAAAiyQAAAHX7AAAAyW4AAAGGoQAAAQA1FQAVwgMVygMs
FaQDFRAVBhUGHBgFAACJTocYBQAAABbbFgAoBQAAiU6HGAUAAAAW2xERAAAA4QHw4AMAAACkAwEINwABAgMEBQYHCAkKCwwNDg8Q
ERITFBUWFxgZGhscHR4fICEiIyQlJicoKSorLC0uLzAxMjM0NTY3ODk6Ozw9Pj9AQUJDREVGR0hJSktMTU5PUFFSU1RVVldYWVpb
XF1eX2BhYmNkZWZnaGlqa2xtbm9wcXJzdHV2d3h5ent8fX5/gIGCg4SFhoeIiYqLjI2Oj5CRkpOUlZaXmJmam5ydnp+goaKjpKWm
p6ipqqusra6vsLGys7S1tre4ubq7vL2+v8DBwsPExcbHyMnKy8zNzs/Q0QAAAAAAABUEGTw1ABgGc2NoZW1hFQQAFQQlAhgIb3Jk
ZXJfaWQAFQ4VChUCGAZhbW91bnQlChUEFRQsXBUEFRQAAAAWpAMZHBksJgAcFQQZNQAGEBkYCG9yZGVyX2lkFQIWpAMWiB8WnhIm
2A0mCBwYCDLrAAAAAAAAGAhh6gAAAAAAABYAKAgy6wAAAAAAABgIYeoAAAAAAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYApAMA
AAAmABwVDhk1AAYQGRgGYW1vdW50FQIWpAMWhBUWkhUmgiMmphIcGAUAAIlOhxgFAAAAFtsWACgFAACJTocYBQAAABbbEREAGSwV
BBUAFQIAFQAVEBUCADwpBhkmAKQDAAAAFow0FqQDJggWsCcAGRwYDEFSUk9XOnNjaGVtYRiMAi8vLy8vOEFBQUFBUUFBQUFBQUFL
QUF3QUJnQUZBQWdBQ2dBQUFBQUJCQUFNQUFBQUNBQUlBQUFBQkFBSUFBQUFCQUFBQUFJQUFBQlFBQUFBQkFBQUFNai8vLzhBQUFF
SEVBQUFBQ0FBQUFBRUFBQUFBQUFBQUFZQUFBQmhiVzkxYm5RQUFBZ0FEQUFFQUFnQUNBQUFBQW9BQUFBQ0FBQUFFQUFVQUFnQUJn
QUhBQXdBQUFBUUFCQUFBQUFBQUFFQ0VBQUFBQ1FBQUFBRUFBQUFBQUFBQUFnQUFBQnZjbVJsY2w5cFpBQUFBQUFJQUF3QUNBQUhB
QWdBQUFBQUFBQUJRQUFBQUFBQUFBQT0AGCBwYXJxdWV0LWNwcC1hcnJvdyB2ZXJzaW9uIDI1LjAuMRksHAAAHAAAAGoCAABQQVIx
""",
}

_data_dir = pathlib.Path("data") / "036-when-chain-order"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

orders = spark.read.parquet((_data_dir / "orders.parquet").resolve().as_posix())
orders.createOrReplaceTempView("orders")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'bucket': '47bc4674c886513137c10ae65ccee0dc7b9b1884f4bec370d1601fd2bed84e18',
                         'orders': '7e1b368c2860b8ded00305453932750e65d410e75b9a25ab62f281b2bd2c26b9',
                         'total_amount': '4f7590da40535b67e5c98462df5dba3704a3aaea6e9c5afbf96440431cfa2b18'},
 'columns': [['bucket', 'string'], ['orders', 'bigint'], ['total_amount', 'decimal(14,2)']],
 'fingerprint': 'c7767f329855253769d48ed750fe2450391da552f58ffb7b66fb95bc878435a7',
 'order_matters': False,
 'row_count': 3}

check = make_check(EXPECTED)

## The situation

Sales wants orders grouped by size: how many small, medium and large orders there
are, and what they are worth.

An AI assistant wrote the job in the answer cell. It runs, the grand total matches
finance to the paisa, and the report went out. The head of enterprise sales then
asked why the company has not taken a single large order all year.

Find the bug and fix it.

## Input tables

### `orders`

One row per order.

| Column | Type | Description |
|---|---|---|
| `order_id` | BIGINT | Unique per order. |
| `amount` | DECIMAL(10,2) | The order value in rupees. Never NULL, always positive. |

## Requirement

Put every order in one bucket and return one row per bucket.

| Bucket | Amount |
|---|---|
| `SMALL` | less than 1000 |
| `MEDIUM` | 1000 or more, and less than 10000 |
| `LARGE` | 10000 or more |

- An order of exactly 1000.00 is `MEDIUM`. An order of exactly 10000.00 is `LARGE`.
- `orders` is the number of orders in the bucket.
- `total_amount` is the sum of their amounts.

## Expected output

One row per bucket: three rows.

| Column | Type | Description |
|---|---|---|
| `bucket` | STRING | `SMALL`, `MEDIUM` or `LARGE`. |
| `orders` | BIGINT | |
| `total_amount` | DECIMAL(14,2) | |

In [ ]:
orders.show(5)

## Your answer

The cell below is the AI-written code. It runs, but its output is wrong. Find the bug and fix it. The tables are available as DataFrames and as views: `orders`.

In [ ]:
from pyspark.sql import functions as F


def solve(spark):
    """Orders and total amount per order-size bucket."""
    orders = spark.table("orders")

    # Classify each order by its amount
    bucket = (
        F.when(F.col("amount") < 1000, "SMALL")
        .when(F.col("amount") >= 1000, "MEDIUM")
        .when(F.col("amount") >= 10000, "LARGE")
    )
    orders = orders.withColumn("bucket", bucket)

    # Aggregate per bucket
    result = orders.groupBy("bucket").agg(
        F.count("*").alias("orders"),
        F.sum("amount").cast("decimal(14,2)").alias("total_amount"),
    )

    return result


result = solve(spark)
result.show()

In [ ]:
check(result)